In [ ]:
import os
import numpy as np
import pandas as pd
# from sklearn.model_selection import train_test_split

In [3]:
## Selecting 6000 junk examples from the astronet triage dataset to be used

import pandas as pd

# 1) Read in the Astronet triage dataset
fpath_triage = 'mnt/tess/astronet/tces-v14-all.csv'
table_triage = pd.read_csv(fpath_triage, header=0, low_memory=False).set_index('Astro ID')


# 2) Pick all B and N examples
b_df = table_triage[table_triage['Final'] == 'B']
n_df = table_triage[table_triage['Final'] == 'N']

b_count = b_df.shape[0]
n_count = n_df.shape[0]

print('B examples:', b_count)
print('N examples:', n_count)

# How many J do we need to reach 6000?
total_needed = 6000
j_needed = total_needed - (b_count + n_count)

print('J examples needed to reach 6000:', j_needed)

# 3) Pick just enough J examples
j_df = table_triage[table_triage['Final'] == 'J']

# If the dataset doesn't have enough J, sample as many as are available
if j_needed > j_df.shape[0]:
    print(f"WARNING: Not enough J examples. Needed {j_needed}, only {j_df.shape[0]} available.")
    j_needed = j_df.shape[0]

j_df = j_df.sample(n=j_needed, random_state=42)  # random_state for reproducibility, optional


# Combine into one dataframe
junk_df = pd.concat([b_df, n_df, j_df], ignore_index=False)

print("B examples:", b_df.shape[0])
print("N examples:", n_df.shape[0])
print("J examples used:", j_df.shape[0])
print("Total junk examples (B+N+J):", junk_df.shape[0])

# -- No disp columns or changing to 'jj' here. That is done later in main code. --


B examples: 784
N examples: 155
J examples needed to reach 6000: 5061
B examples: 784
N examples: 155
J examples used: 5061
Total junk examples (B+N+J): 6000


In [4]:
junk_df

,TIC ID,Final,Decision,av,md,ch,as,mk,et,dm,...,Split,Year,MaxT,MinT,File,disp_E,disp_J,disp_N,disp_S,disp_B
Astro ID,,,,,,,,,,,,,,,,,,,,,
21,119088593,B,B,B,E,B,NaN,NaN,NaN,NaN,...,train,1,1682.343934,1653.927253,tess2019247000000-0000000119088593-111-cr_llc....,0,0,0,0,1
22,143769346,B,NaN,B,B,B,NaN,NaN,NaN,NaN,...,train,1,1682.343158,1653.926401,tess2019247000000-0000000143769346-111-cr_llc....,0,0,0,0,1
24,63343395,B,B,E,B,B,NaN,NaN,NaN,NaN,...,train,1,1682.344122,1653.927462,tess2019247000000-0000000063343395-111-cr_llc....,0,0,0,0,1
46,407089973,B,NaN,B,B,B,NaN,NaN,NaN,NaN,...,test,1,1682.344736,1624.969731,tess2019247000000-0000000407089973-111-cr_llc....,0,0,0,0,1
83,350137597,B,B,B,S,J,NaN,NaN,NaN,NaN,...,val,1,1682.343679,1325.323219,tess2019247000000-0000000350137597-111-cr_llc....,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11329,441735063,J,NaN,NaN,J,J,J,J,NaN,NaN,...,train,2,2009.280926,1683.365370,tess2020262000000-s0025-0000000441735063-1111-...,0,1,0,0,0
10730,330093334,J,NaN,NaN,J,J,J,J,NaN,NaN,...,train,2,1763.305101,1711.384083,tess2020262000000-s0016-0000000330093334-1111-...,0,1,0,0,0
5607,340000943,J,NaN,J,J,J,J,NaN,NaN,NaN,...,train,1,1682.345341,1381.717938,tess2019247000000-0000000340000943-111-cr_llc....,0,1,0,0,0


In [6]:
# Generate a .txt file with the TIC IDs (important for generating the fits files)

junk_df['TIC ID'].to_csv('junk_data_2025_TIC_list.txt', index=False, header=False)

In [9]:
# Count how many rows in junk_df contain "mk" in their File column
mk_df = junk_df[junk_df['File'].str.contains('mk', na=False)]
print(f"Number of examples with 'mk' in File: {mk_df.shape[0]}/{junk_df.shape[0]}")


Number of examples with 'mk' in File: 569/6000


In [ ]:
# Part 2: adding the new filenames to the dataframe
